<a href="https://colab.research.google.com/github/Andraiss/Arnet/blob/master/4_sizing_procedure_examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Design graph and optimization with python

*Written by Marc Budinger (INSA Toulouse), Toulouse, France*

This notebook continues the example started [earlier](./2-optimization_examples.ipynb). It aims to show how a design graph can be used to structure the optimisation problem.

For interested readers, more information can be found in :
> Delbecq, S., Budinger, M., Ochotorena, A., Reysset, A., & Defaÿ, F. (2020). Efficient sizing and optimization of multirotor drones based on scaling laws and similarity models. Aerospace Science and Technology, 102, 105873.

### Objectives and specifications

We want to optimize the selection of a TVC motor/reducer set: dynamics are high and motor inertia effect is not negligeable.

![EMA](https://github.com/SizingLab/sizing_course/blob/main/classes/Lecture4_new/assets/images/TVC_EMA.png?raw=1)

The sizing scenario is a sinusoïdal displacement with a constant force and an inertia effect:

| Specification |  |
|---|---|
| static force | 40 kN |  
| Inertia | 800 kgg |
|Displacement magnitude | 10 mm |
| Frequency | 5 Hz |



We first define the specifications and assumptions for the sizing:

In [1]:
from math import pi

# Specifications
Xmax = 10e-3  # [m]
f = 5  # [Hz]
w = 2*pi*5 # [rad/s] angular frequency

Xpmax = w*Xmax  # [m/s] max linear speed
Xppmax = w**2*Xmax  # [m/s²] max linear acceleration

M = 800 # [kg] Equivalent mass inertia of the nozzle
Fstat = 40e3 # [N] static force

A reference example of motor is:

| Characteristics |  |
|---|---|
| Nom. torque | 3.14 N.m|
| Max. torque | 13.4	N.m	 |  
| Max. speed | 7200	rpm |
| Inertia  | 2.90E-04	kg.m² |
| Mass | 3.8 	kg |

We assume here that :
- the pitch $p$ of the roller screw is 10 mm/rev.
- The efficiency of mechanical power transmission is 100%.

The objective is to select the reduction ratio of the gear reducer in order to minimize the mass of the motor.

We then define the main characteristics of components and the reference parameters for the scaling laws:

In [2]:
# Assumptions
pitch = 10e-3 / 2 / 3.14  # [m/rad]
nu_screw = 1  # [-]

# Reference motor
T_mot_nom_ref = 3.14 # [N.m] Max torque
T_mot_max_ref = 13.4  # [N.m] Rated torque
W_mot_max_ref = 754  # [rad/s] max speed
J_mot_ref = 2.9e-4 / 2  # [kg.m²] Inertie
M_mot_ref = 3.8  # [kg] Mass


## Design graph

The objective is to select the motor and the reduction ratio of a gear reducer in order to minimize the mass of the motor.


```{exercise}
:label: DesignGraph

- Draw the design graph and highlight the main problems.
- Propose a solution which reduce the complexity of the optimization problem compared to [previous solution](./2-optimization_examples.ipynb).
- Define optimization variables and the set of constraints
```

```{solution} DesignGraph
:label: DesignGraphSolution
:class: dropdown
       
The design graph of this sizing problem is:
![Algebraic Loop](./assets/images/AlgebraicLoop.png)
with  
- equation `1` the sizing scenario linking force and acceleration of the load to the maximal electromagnetic torque of the motor.
- equations `2` and `3` scaling laws which enable to estimate inertia and mass of the motor.  

The main problem is the existance of an algebraic loop. Here the choice of a motor, based on its max torque depends on its own inertia, which means we have to know what motor to chose before choosing it.  
This  algebraic loop can be broken by admitting that the inertia torque is meaningless to evaluate the max torque, which allows us to choose a motor.  
The global torque with inertia can subsequently be evaluated and compared to the torque of the motor we selected. To respect this last constraint, we add an oversizing constraint to the first sizing process, the solver  will chose by itself the value of this coefficient, which is at least 1 (in  the case of negligible inertia effects).         
        
```

## Sizing code

The sizing code is defined here in a function which can give an evaluation of the objective and of the constraints function of design variables.

In [25]:
def sizing_code(param, arg="print"):
    #  Design variable
    #N = param[0]  # Reduction ratio of the reducer
    #k_mot = param[1]  # oversizing coefficient
    k_MTOW = param[0]
    k_ND = param[1]
    k_bat = param[2]
    beta_pro = param[3]

    # -----------------------
    nu_motor = 0.8
    nu_esc = 0.95
    M_payload = 1
    RatSM_Load = 1
    rho = 1.118

    ND_max = 44.5
    a_to = 1
    N_pro_arm = 1
    N_arm = 4

    # Calcul de la masse totale
    MTOW_est = k_MTOW * M_payload
    N_pro = N_pro_arm * N_arm

    # Effort par helice au decollage
    F_pro_to = MTOW_est * (9.81 + a_to) / N_pro
    F_pro_hov = (MTOW_est * 9.81) / N_pro

    # Choix heleice sur decollage
    Cp_pro = -0.00148 + 0.0972 * beta_pro
    Ct_pro = 0.0438 + 0.1414 * beta_pro
    D_pro = (F_pro_to / (Ct_pro * rho * (ND_max / k_ND)**2 ))**0.5
    n_pro_to = 44.5 / (k_ND * D_pro)

    # Consommation en hover et energie batterie
    n_pro_hov = (F_pro_hov / (Ct_pro * rho * D_pro**4))**0.5
    P_pro_hov = Cp_pro * rho * (n_pro_hov**3) * (D_pro**5)
    E_flight = P_pro_hov * 1200 * N_pro / (nu_esc * nu_motor)
    E_bat = E_flight * k_bat / (1 - 0.2)
    P_pro_to = Cp_pro * rho * (n_pro_to**3) * (D_pro**5)

    # Calcul des masses
    M_pro = 0.014999 * (D_pro / 0.2794)**3
    M_bat = 0.273 * (E_bat / 133200) * 0.5
    P_bat_max = (1850 * M_bat) / 0.273
    P_flight = P_pro_to * N_pro/(nu_motor*nu_esc)
    MTOW = M_pro * N_pro + M_bat + M_payload * (1 + RatSM_Load)

    # --------------------------------------

    # --------------------------------------
    # Objectives and constrants calculations

    # Objective = motor mass
    objective = MTOW

    # Constraints (should be >=0)
    C1 = MTOW_est - MTOW
    C2 = P_bat_max - P_flight

    # --------------------------------------
    # Objective and constraints
    if arg == "objective":
        return objective
    if arg == "objectiveP":
        if ((C1 < 0.0) | (C2 < 0.0)):
            # If constraints are not respected we penalize
            # the objective for contraint less algorithms
            return objective * 1e5
        else:
            return objective
    elif arg == "print":
        print("Objective:")
        print("     Total mass = %.2f kg" % MTOW)
        print("     C1 = %.2f kg" % C1)
        print("     C2 = %.2f W" % C2)
        print("     D_pro = %.2f m"% D_pro)
        print("     M_bat = %.2f kg"% M_bat)
        print("     M_propellers = %.2f kg"% (M_pro*4))
        print("     M_strcuture = %.2f kg"% M_payload*RatSM_Load)

    elif arg == "constraints":
        return [C1, C2]
    else:
        raise TypeError(
            "Unknown argument for sizing_code: use 'print', 'objective', 'objectiveP' or 'contraints'"
        )

## Optimization with SLSQP algorithm


We will now use the [opmization algorithms](https://docs.scipy.org/doc/scipy/reference/optimize.html) of the Scipy package to solve and optimize the configuration. We will first use the [SLSQP](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-slsqp.html) algorithm without explicit expression of the gradient (Jacobian).

The first step is to give an initial value of optimisation variables:

In [26]:
import numpy as np

# Initial values vector for design variables
parameters = np.array((1,1,1,0.4))

We can print of the characterisitcs of the problem before optimization with the intitial vector of optimization variables:

In [27]:
# Initial characteristics before optimization
print("-----------------------------------------------")
print("Initial characteristics before optimization :")

sizing_code(parameters, "print")
print("-----------------------------------------------")

-----------------------------------------------
Initial characteristics before optimization :
Objective:
     Total mass = 2.32 kg
     C1 = -1.32 kg
     C2 = 1888.23 W
     D_pro = 0.11 m
     M_bat = 0.31 kg
     M_propellers = 0.00 kg
     M_strcuture = 1.00 kg
-----------------------------------------------


We can see that constraints are not respected.

Then we can solve the problem and print of the optimized solution:

In [28]:
import scipy
import time

# Resolution of the problem

contrainte = lambda x: sizing_code(x, "constraints")
objectif = lambda x: sizing_code(x, "objective")

start = time.time()
result = scipy.optimize.fmin_slsqp(
    func=objectif,
    x0=parameters,
    bounds=[
        (1, 10),
        (1, 10),
        (1, 10),
        (0.3, 0.6)
    ],
    f_ieqcons=contrainte,
    iter=1000,
    acc=1e-5,
)
end = time.time()

# Final characteristics after optimization
print("-----------------------------------------------")
print("Final characteristics after optimization :")

sizing_code(result, "print")
print("-----------------------------------------------")
print("Calculation time:\n", end - start, "s")

Optimization terminated successfully    (Exit mode 0)
            Current function value: 2.475478582463493
            Iterations: 6
            Function evaluations: 30
            Gradient evaluations: 6
-----------------------------------------------
Final characteristics after optimization :
Objective:
     Total mass = 2.48 kg
     C1 = -0.00 kg
     C2 = 2148.24 W
     D_pro = 0.35 m
     M_bat = 0.36 kg
     M_propellers = 0.12 kg
     M_strcuture = 1.00 kg
-----------------------------------------------
Calculation time:
 0.004442453384399414 s


```{exercise}
:label: Resolution speed

- Compare the speed of execution of this optimization with the [previous solution](./2-optimization_examples.ipynb). This speed can be evaluated with the number of calls to the function (the sizing code).
- Explain the observed discrepancy.
```